### Step 1: Install Dependencies
First, make sure you have MLflow and scikit-learn installed. You can install them using pip:

In [1]:
#!pip3 install mlflow scikit-learn

### Step 2: Import Libraries
In your Python script, import the necessary libraries:

In [2]:
import mlflow
import mlflow.sklearn
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from urllib.parse import urlparse

import numpy as np
import pandas as pd
from sklearn.linear_model import ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature
mlflow.autolog() 


2026/10/29 02:55:34 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.


### Step 3: Load and Prepare Data
Load a dataset from scikit-learn, split it into training and testing sets, and perform any necessary data preprocessing:

In [3]:
data = load_diabetes()
X = data.data
y = data.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### Step 4: Train and Log the Model
Train a simple linear regression model using scikit-learn and log it using MLflow:

In [4]:
with mlflow.start_run() as run:
    # Create and train a linear regression model
    lr = LinearRegression()
    lr.fit(X_train, y_train)

    predictions = lr.predict(X_train)
    signature = infer_signature(X_train, predictions)

    tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme

    # Model registry does not work with file store
    if tracking_url_type_store != "file":
        # Register the model
        # There are other ways to use the Model Registry, which depends on the use case,
        # please refer to the doc for more information:
        # https://mlflow.org/docs/latest/model-registry.html#api-workflow
        mlflow.sklearn.log_model(
            lr, "model", registered_model_name="RandomForest", signature=signature
        )
    else:
        mlflow.sklearn.log_model(lr, "model", signature=signature)

print(f"Run ID: {run.info.run_id}")

2026/10/29 02:55:37 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


2026/10/29 02:55:43 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Run ID: 105b023258d04289bc33301f4faa2adc


Successfully registered model 'RandomForest'.
Created version '1' of model 'RandomForest'.


This code initializes an MLflow run, trains the model, and logs it with the name "model."

### Step 5: Save the MLflow Artifacts
By default, MLflow logs the model into an artifact store. You can specify the artifact location in your MLflow server configuration or use the default location. You can also set the environment variable MLFLOW_TRACKING_URI to point to your MLflow server.

### Step 6: Serve the Model
To serve the model using MLflow, you can use the mlflow models serve command. First, make sure that MLflow is running as a server (e.g., mlflow server).

Then, you can start serving your model:

### In case of Error

In [5]:
# https://github.com/pyenv/pyenv/wiki#suggested-build-environment

In [6]:
# !rm -rf '/home/ramin/.pyenv'

In [7]:
# !curl https://pyenv.run | bash
# !python -m  pip install virtualenv
# !PATH="$HOME/.pyenv/bin:$PATH"
# !export PYENV_ROOT="$HOME/.pyenv"

### In-case of dependeny error

In [8]:
# mlflow.pyfunc.get_model_dependencies('mlruns/0/e6f6f958a20c407f9d485ba5fb24f6c2/artifacts/model')

In [9]:
# !pip3 install -r mlruns/0/aea73ab36d544af29927cf26199b8888/artifacts/model/requirements.txt

### To run on a severate bash screen within the same venv

In [ ]:
!mlflow models serve --env-manager=local -m runs:/{run.info.run_id}/model -h 127.0.0.1 -p 5001

### Step 7: Make Predictions
You can make predictions by sending HTTP POST requests to the model server. Here's an example using Python's requests library:

In [10]:
X_test.shape

(89, 10)

In [11]:
import requests
import json

url = 'http://127.0.0.1:5001/invocations'

data = {
    "columns": data['feature_names'],
    "instances": X_test.tolist()
}

response = requests.post(url, json=data)
predictions = response.json()

print(predictions)

{'predictions': [139.5475584037962, 179.51720835342775, 134.03875571890114, 291.4170292522082, 123.7896587223961, 92.17234650105051, 258.23238898921295, 181.33732057060732, 90.22411310941467, 108.63375858007933, 94.1386574363886, 168.4348635797548, 53.50478879609557, 206.63081659296427, 100.12925868961602, 130.66657085052702, 219.53071499182568, 250.7803234033335, 196.3688346018638, 218.57511814821783, 207.3505018197475, 88.48340941420328, 70.43285917378932, 188.95914235039007, 154.88681619689794, 159.361701217919, 188.31263362697558, 180.39094032857128, 47.99046560870016, 108.97453871427219, 174.77897632897373, 86.36406655640151, 132.95761214676352, 184.53819482628643, 173.83220911339018, 190.35858491915613, 124.4156176013922, 119.65110656071295, 147.95168682418492, 59.05405241472812, 71.62331856151584, 107.68284703885507, 165.4536545804207, 155.00975931051346, 171.04799095638182, 61.45761356101795, 71.66672580728225, 114.96732205941936, 51.5797552304239, 167.57599528011522, 152.52291

The url variable, in the context of the code provided, represents the Uniform Resource Locator (URL) that is used to specify the location or address of a web resource. In this case, it points to a specific endpoint on a local web server:

* http:// indicates the protocol being used, which is Hypertext Transfer Protocol (HTTP). HTTP is the foundation of data communication on the World Wide Web.

* localhost is a special hostname that typically refers to the current device or computer where the code is running. It is often used to access services running on the same machine.

* 5000 is the port number. Ports are used to differentiate between different services or processes running on the same machine. Port 5000 is a commonly used port for running web servers locally.

* /invocations is the path or endpoint on the web server. In the context of machine learning deployment, this path often represents an API endpoint for making predictions using a deployed machine learning model. In this specific case, it's likely that the web server running on localhost:5000 has an endpoint called /invocations that allows you to send data to the deployed model and receive predictions in response.

So, the url variable is essentially storing the URL of the API endpoint where you can send a POST request with data to obtain predictions from a locally deployed machine learning model.